In [ ]:
# Make sure to run every command in README.md first to setup the environment #

# Parameters for preparing the data set #

In [ ]:
# Image dimensions to scale images and masks!

input_size_x = 348;
input_size_y = 348;

assert input_size_x >= 32 and input_size_x < 4096, 'Dimensions for scaling should be between 32 and 4096'
assert input_size_y >= 32 and input_size_y < 4096, 'Dimensions for scaling should be between 32 and 4096'

# Two visit, segmentation

# Train, validation and test split percentages!TODO [Validation is 1.0] Assert if 1.0, Error messages

train_percentage = 0.5;
test_percentage = 0.4;

assert train_percentage > 0.0, 'Train percentage needs to be larger'
assert (train_percentage+test_percentage) <= 0.9, 'Train percentage + validation needs to be less than or equal 0.9'

validation_percentage = 1.0-(train_percentage+test_percentage);

print ("Percentages for splits train/val/test", round(train_percentage,2), round(validation_percentage,2), round(test_percentage,2))

# Seed for selecting samples

seed = 1

# There are four gradings to select from [Options 1 to 3 are manual gradings] and -1 is for automatic graded images

grading = 1;

assert grading in [-1,1,2,3], "Wrong grading number"

# Output folder to save images and masks for training

output = "./output/";

# Output in RGB or grayscale format for the masks

masks_in_grayscale = True;

# Unzip dataset #

In [ ]:
# Place the zip file within folder "Prepare_AI4Eyes_Dataset_01"

#!unzip -q AMD_DME_Normals.zip

# Prepare data set for training #

In [ ]:
import os
from natsort import natsorted
import cv2
import numpy as np
import pathlib

if(grading != -1):
    rootdir = '../../AMD_DME_Normals/Images/Images_Manual/'
else:
    rootdir = '../../AMD_DME_Normals/Images/Images_Automatic/'

allImages_dir = []

if(masks_in_grayscale):
    if(grading != -1):
        replaceStr = "/Masks/Masks_Manual/Grading_"+str(grading)+"/"
        searchStr = "/Images/Images_Manual/"
    else:
        replaceStr = "/Masks/Masks_Automatic/Grading"+"/"
        searchStr = "/Images/Images_Automatic/"
else:
    if(grading != -1):
        replaceStr = "/Masks/Masks_Manual_RGB/Grading_"+str(grading)+"/"
        searchStr = "/Images/Images_Manual/"
    else:
        replaceStr = "/Masks/Masks_Automatic_RGB/Grading"+"/"
        searchStr = "/Images/Images_Automatic/"
        
for subdir, dirs, files in os.walk(rootdir):
    for file in files:
        if(".png" in file or ".PNG" in file or ".TIFF" in file or ".TIF" in file or ".jpeg" in file):
            allImages_dir.append(subdir)
            
allImages_dir = np.unique(allImages_dir)
allImages_dir = natsorted(list(allImages_dir))

# Shuffle list of directories with the seed #

In [ ]:
np.random.seed(seed)
np.random.shuffle(allImages_dir)

In [ ]:
from0 = 0; to0 = int(len(allImages_dir)*train_percentage);
from1 = to0; to1 = from1+int(len(allImages_dir)*validation_percentage);
from2 = to1; to2 = len(allImages_dir);

print (from0, to0)
print (from1, to1)
print (from2, to2)

allImages_dir_tr = allImages_dir[from0:to0];
allImages_dir_v = allImages_dir[from1:to1];
allImages_dir_te = allImages_dir[from2:to2];

print (len(allImages_dir_tr))
print (len(allImages_dir_v))
print (len(allImages_dir_te))

allImages_tr = []
masks_tr = [];

allImages_v = []
masks_v = [];

allImages_te = []
masks_te = [];

for subdir, dirs, files in os.walk(rootdir):
    for file in files:
        if(".png" in file or ".PNG" in file or ".TIFF" in file or ".TIF" in file or ".jpeg" in file):
            if(subdir in allImages_dir_tr):
                allImages_tr.append(os.path.join(subdir, file))
                masks_tr.append((os.path.splitext(allImages_tr[-1])[0]+'.png').replace(searchStr, replaceStr))
            elif(subdir in allImages_dir_v):
                allImages_v.append(os.path.join(subdir, file))
                masks_v.append((os.path.splitext(allImages_v[-1])[0]+'.png').replace(searchStr, replaceStr))
            elif(subdir in allImages_dir_te):
                allImages_te.append(os.path.join(subdir, file))
                masks_te.append((os.path.splitext(allImages_te[-1])[0]+'.png').replace(searchStr, replaceStr))

In [ ]:
assert (len(allImages_dir_tr)+len(allImages_dir_v)+len(allImages_dir_te)) == len(allImages_dir), "Number of folders for the splits are wrong"

# Sort files by name #

In [ ]:
allImages_tr = natsorted(allImages_tr); masks_tr = natsorted(masks_tr);
allImages_v = natsorted(allImages_v); masks_v = natsorted(masks_v);
allImages_te = natsorted(allImages_te); masks_te = natsorted(masks_te);

In [ ]:
len(allImages_tr), len(masks_tr)

In [ ]:
# Quick check if image and msk is correspond to each other

print (allImages_tr[-1])
print (masks_tr[-1])

# Create output folder #

In [ ]:
pathlib.Path(output).mkdir(parents=True, exist_ok=True)

In [ ]:
import imageio

# Scale images and masks #

In [ ]:
# Training images

train_img = []; train_msk = [];

for i in range(0, len(allImages_tr)):
    train_img.append(cv2.resize(imageio.imread(allImages_tr[i]), dsize=(input_size_x, input_size_y)));
    train_msk.append(cv2.resize(imageio.imread(masks_tr[i]), dsize=(input_size_x, input_size_y),interpolation=cv2.INTER_NEAREST))

In [ ]:
# Validation images

valid_img = []; valid_msk = [];

for i in range(0, len(allImages_v)):
    valid_img.append(cv2.resize(imageio.imread(allImages_v[i]), dsize=(input_size_x, input_size_y)));
    valid_msk.append(cv2.resize(imageio.imread(masks_v[i]), dsize=(input_size_x, input_size_y),interpolation=cv2.INTER_NEAREST))

In [ ]:
# Testing images

test_img = []; test_msk = [];

for i in range(0, len(allImages_te)):
    test_img.append(cv2.resize(imageio.imread(allImages_te[i]), dsize=(input_size_x, input_size_y)));
    test_msk.append(cv2.resize(imageio.imread(masks_te[i]), dsize=(input_size_x, input_size_y),interpolation=cv2.INTER_NEAREST))

# Save images and masks #

In [ ]:
# Training images

pathlib.Path(output+"/train_img").mkdir(parents=True, exist_ok=True)
pathlib.Path(output+"/train_msk").mkdir(parents=True, exist_ok=True)

for i in range(0, len(train_img)):
    imageio.imsave(output+"/train_img/"+str(i).zfill(4)+".png", train_img[i]);
    imageio.imsave(output+"/train_msk/"+str(i).zfill(4)+".png", train_msk[i]);

In [ ]:
# Validation images

pathlib.Path(output+"/valid_img").mkdir(parents=True, exist_ok=True)
pathlib.Path(output+"/valid_msk").mkdir(parents=True, exist_ok=True)

for i in range(0, len(valid_img)):
    imageio.imsave(output+"/valid_img/"+str(i).zfill(4)+".png", valid_img[i]);
    imageio.imsave(output+"/valid_msk/"+str(i).zfill(4)+".png", valid_msk[i]);

In [ ]:
# Testing images

pathlib.Path(output+"/test_img").mkdir(parents=True, exist_ok=True)
pathlib.Path(output+"/test_msk").mkdir(parents=True, exist_ok=True)

for i in range(0, len(test_img)):
    imageio.imsave(output+"/test_img/"+str(i).zfill(4)+".png", test_img[i]);
    imageio.imsave(output+"/test_msk/"+str(i).zfill(4)+".png", test_msk[i]);

# Quick check if pairs of images and masks are ok #

In [ ]:
import random

In [ ]:
ids = random.sample(range(0, len(train_img)), 50);

In [ ]:
from matplotlib import pylab as plt
fig, ax = plt.subplots(6,5,figsize=(20,20))

fig.suptitle('Randomly selected samples');

cnt = 0;
for i in range(0, 6, 2):
    for j in range(0, 5, 1):
        ax[i,j].imshow(train_img[ids[cnt]],cmap="gray");
        ax[i+1,j].imshow(train_msk[ids[cnt]]);
        cnt += 1;
plt.tight_layout();
plt.show();